In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


sample = {
    "en": "Open the consumption model containing the measures and attributes you want to include in your perspective, and click the Perspectives tab.",
    "proper_terms": {
        "consumption model": "Verbrauchsmodell"
    },
    "random_terms": {
        "include": "aufnehmen",
        "want": "möchten"
    }
}


prompt = f"""
You are a translation assistant.


Translate the English text to German.


Rules:
1. Output only in this format: <deu> ... </deu>
2. Use the terminology mappings exactly as provided.
3. Do not explain anything.


Terminology:
{sample["proper_terms"]}
{sample["random_terms"]}


Input:
<en> {sample["en"]} </en>
"""


messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]


response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]